<a href="https://colab.research.google.com/github/Hem1144/AI-ML/blob/main/is_report2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install needed libraries

!pip install -q -U \
  langchain \
  langchain-community \
  langchain-google-genai \
  sentence-transformers \
  faiss-cpu \
  huggingface_hub

**Set Up API Keys (Gemini + Hugging Face)**

In [ ]:
# Configure API keys

import os, getpass

# Gemini (Google AI Studio / Gemini API)
os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Gemini (Google AI) API key: ")

# Hugging Face (optional but recommended for faster / authenticated model downloads)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass.getpass("Enter your Hugging Face API token (or press Enter if none): ")

Enter your Gemini (Google AI) API key: ··········
Enter your Hugging Face API token (or press Enter if none): ··········


**Imports and Basic LLM Setup**

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage

from dataclasses import dataclass

# Use Gemini 2.5 Flash
GEMINI_MODEL_NAME = "gemini-2.5-flash"

# Create a single shared LLM object
llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL_NAME,
    temperature=0.4,
)

**Build a Tiny Knowledge Base (RAG with HuggingFace + FAISS)**

In [ ]:
# Create some example documents for RAG

raw_documents = [
    """
    Multi-agent intelligent systems involve several specialized agents
    that collaborate to solve complex tasks. Typical roles include:
    - Planner / Coordinator
    - Research / Retrieval Agent (RAG)
    - Developer / Solver
    - Evaluator / Critic
    These systems are useful for tasks like project selection,
    document analysis, or automated report generation.
    """,
    """
    Good AI project ideas often combine:
    - A real-world problem (health, education, finance, environment)
    - A suitable dataset
    - Appropriate AI methods (ML, Deep Learning, NLP, RAG)
    Students should choose projects aligned with their interests and skill levels.
    """,
    """
    Retrieval-Augmented Generation (RAG) combines:
    - An embedding model (e.g., Hugging Face sentence transformers)
    - A vector store (e.g., FAISS, Chroma)
    - A language model (e.g., Gemini)
    The system retrieves relevant chunks of text and passes them to the LLM
    to generate grounded, accurate answers.
    """,
]

# Split docs into smaller chunks for better retrieval
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
)

docs = text_splitter.create_documents(raw_documents)

len(docs), docs[0].page_content[:200]

(3,
 'Multi-agent intelligent systems involve several specialized agents\n    that collaborate to solve complex tasks. Typical roles include:\n    - Planner / Coordinator\n    - Research / Retrieval Agent (RAG')

Now embed those chunks with a Hugging Face embedding model and store them in FAISS.

In [ ]:
# Build embeddings + vector store (RAG)

# A small, fast embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

hf_embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name
)

# Build FAISS vector store from our small knowledge base
vectorstore = FAISS.from_documents(docs, hf_embeddings)

# Create a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

**Helper to Call Gemini with a System Prompt**

In [ ]:
# Helper function for LLM calls

def run_llm(system_prompt: str, user_content: str, temperature: float = 0.4) -> str:
    """
    Run Gemini with a system prompt + user content using LangChain's ChatGoogleGenerativeAI.
    """
    model = llm.bind(temperature=temperature)
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_content),
    ]
    response = model.invoke(messages)
    return response.content.strip()

**Define Agent Class and Specialized Agents**

In [ ]:
# Generic Agent abstraction

@dataclass
class Agent:
    name: str
    role: str
    expertise: str
    style: str = "Think step by step. Explain your reasoning clearly."

    def system_prompt(self) -> str:
        return (
            f"You are {self.name}, acting as a {self.role}.\n"
            f"Your expertise: {self.expertise}.\n"
            f"{self.style}"
        )

    def run(self, task: str, context: str = "", temperature: float = 0.4) -> str:
        """
        Execute the agent for a task with optional shared context.
        """
        user_prompt = (
            f"--- TASK ---\n{task}\n\n"
            f"--- CONTEXT (from other agents or documents) ---\n{context}\n\n"
            "Now produce your best, detailed response for your part of the problem."
        )
        return run_llm(self.system_prompt(), user_prompt, temperature=temperature)

Now define four specific agents with different reasoning styles

In [ ]:
# Define specialized agents

planner_agent = Agent(
    name="Planner Agent",
    role="Task Decomposition and Strategy Expert",
    expertise=(
        "Breaking complex user goals into clear, ordered sub-tasks, "
        "identifying dependencies and proposing a good overall strategy."
    ),
    style=(
        "First restate the goal concisely. Then write assumptions. "
        "Then produce a numbered step-by-step plan, with bullet details under each step."
    ),
)

research_agent = Agent(
    name="Research Agent (RAG)",
    role="Retrieval-Augmented Reasoning Expert",
    expertise=(
        "Using retrieved documents and background knowledge to reason deeply "
        "about AI, multi-agent systems, and project design."
    ),
    style=(
        "Use the provided retrieved context carefully. "
        "Quote or paraphrase important points, and connect them to the task."
    ),
)

developer_agent = Agent(
    name="Developer Agent",
    role="Implementation and Explanation Expert",
    expertise=(
        "Writing clean, well-commented Python code, "
        "and explaining how it works step by step."
    ),
    style=(
        "First outline the design, then show code blocks with comments, "
        "then explain how to run and adapt the code."
    ),
)

reviewer_agent = Agent(
    name="Reviewer Agent",
    role="Critical Reviewer and Refiner",
    expertise=(
        "Checking reasoning and code for correctness, clarity, robustness, "
        "and alignment with the user's goal and marking criteria."
    ),
    style=(
        "Be constructive but critical. Point out weaknesses, then provide "
        "an improved final answer. If the solution is good, confirm and suggest small refinements."
    ),
)

**RAG Helper for the Research Agent**

In [ ]:
# RAG helper for research agent

def rag_research(user_goal: str, plan: str, extra_question: str = "") -> str:
    """
    1. Build a query combining the goal and plan.
    2. Retrieve relevant chunks from vector store.
    3. Let the ResearchAgent reason using those chunks as context.
    """
    query = f"{user_goal}\n\nPlan:\n{plan}\n\nExtra question:\n{extra_question}"
    retrieved_docs = retriever.invoke(query)

    context_text = "\n\n--- RETRIEVED DOC ---\n".join(
        [d.page_content for d in retrieved_docs]
    )

    task = (
        "Use the retrieved context to design an intelligent multi-agent solution "
        "and explain important concepts relevant to the user's goal. "
        "Highlight how RAG, multi-agent collaboration, and Gemini can work together."
    )

    return research_agent.run(
        task=task,
        context=context_text + "\n\nUSER GOAL:\n" + user_goal + "\n\nPLAN:\n" + plan,
        temperature=0.4,
    )

**Multi-Agent Orchestrator**

In [ ]:
# Multi-agent orchestrator

class MultiAgentSystem:
    """
    Pipeline:
      1) Planner Agent: break down the task
      2) Research Agent (RAG): use docs + reasoning for design
      3) Developer Agent: produce detailed code / explanation
      4) Reviewer Agent: critique and refine final answer
    """

    def __init__(self, planner, researcher, developer, reviewer):
        self.planner = planner
        self.researcher = researcher
        self.developer = developer
        self.reviewer = reviewer

    def solve(self, user_goal: str) -> dict:
        # 1. Planning
        print("[1/4] Planner Agent is thinking...")
        plan = self.planner.run(
            task="Break down the user's goal into a detailed, numbered plan.",
            context=user_goal,
            temperature=0.3,
        )

        # 2. Research with RAG
        print("[2/4] Research Agent (RAG) is reasoning with retrieved docs...")
        research = rag_research(
            user_goal=user_goal,
            plan=plan,
            extra_question="Focus on multi-agent design and how to structure the Colab code.",
        )

        # 3. Development / Implementation
        print("[3/4] Developer Agent is generating the implementation...")
        dev_task = (
            "Using the PLAN and RESEARCH, write a clear description of the multi-agent system. "
            "Then provide Python code (for Google Colab) that implements it using "
            "LangChain + HuggingFace embeddings + Gemini. "
        )
        implementation = self.developer.run(
            task=dev_task,
            context=(
                f"USER GOAL:\n{user_goal}\n\n"
                f"PLAN:\n{plan}\n\n"
                f"RESEARCH:\n{research}"
            ),
            temperature=0.45,
        )

        # 4. Review / Final refinement
        print("[4/4] Reviewer Agent is reviewing the whole solution...")
        review_task = (
            "Review the PLAN, RESEARCH, and IMPLEMENTATION for correctness, clarity, "
            "and robustness. Fix any mistakes in the code if needed, and provide a final, "
            "clean answer. If the code looks good, still restate the final structure clearly."
        )
        review_and_final = self.reviewer.run(
            task=review_task,
            context=(
                f"USER GOAL:\n{user_goal}\n\n"
                f"PLAN:\n{plan}\n\n"
                f"RESEARCH:\n{research}\n\n"
                f"IMPLEMENTATION (from Developer):\n{implementation}"
            ),
            temperature=0.35,
        )

        return {
            "plan": plan,
            "research": research,
            "implementation": implementation,
            "review_and_final": review_and_final,
        }

In [ ]:
# Create the multi-agent system instance

multi_agent = MultiAgentSystem(
    planner=planner_agent,
    researcher=research_agent,
    developer=developer_agent,
    reviewer=reviewer_agent,
)

In [ ]:
!pip install -q ipywidgets markdown beautifulsoup4

import markdown
from bs4 import BeautifulSoup

def markdown_to_text(md: str) -> str:
    """
    Convert Markdown text to clean plain text.
    """
    if not md:
        return ""
    html = markdown.markdown(md)
    soup = BeautifulSoup(html, "html.parser")
    return soup.get_text(separator="\n").strip()


In [ ]:
from IPython.display import display
import ipywidgets as widgets

# UI COMPONENTS

title = widgets.HTML(
    """
    <h2 style="text-align:center;">Multi-Agent Intelligent System</h2>
    <p style="text-align:center;">
        AI Project Selection & Intelligent System Design Assistant
    </p>
    <hr>
    """
)

query_input = widgets.Textarea(
    placeholder="Enter your AI-related query here...\n\nExample:\nDesign a multi-agent system for healthcare diagnosis.",
    layout=widgets.Layout(width="100%", height="140px")
)

run_button = widgets.Button(
    description="Generate Response",
    button_style="primary",
    icon="play"
)

output_area = widgets.Output(
    layout={'border': '1px solid black', 'padding': '10px'}
)

# BUTTON LOGIC

def on_run_clicked(b):
    with output_area:
        output_area.clear_output()
        print("Processing your request... Please wait.\n")

        # Run multi-agent system
        results = multi_agent.solve(query_input.value)

        # Parse markdown → plain text
        planner_text = markdown_to_text(results["plan"])
        research_text = markdown_to_text(results["research"])
        developer_text = markdown_to_text(results["implementation"])
        reviewer_text = markdown_to_text(results["review_and_final"])

        # Display formatted output
        print("=" * 80)
        print("FINAL SYSTEM OUTPUT")
        print("=" * 80)

        print("\n--- PLANNER AGENT OUTPUT ---\n")
        print(planner_text)

        print("\n" + "-" * 80)
        print("\n--- RESEARCH AGENT (RAG) OUTPUT ---\n")
        print(research_text)

        print("\n" + "-" * 80)
        print("\n--- DEVELOPER AGENT OUTPUT ---\n")
        print(developer_text)

        print("\n" + "-" * 80)
        print("\n--- REVIEWER AGENT (FINAL ANSWER) ---\n")
        print(reviewer_text)

        print("\n" + "=" * 80)
        print("END OF RESPONSE")
        print("=" * 80)

run_button.on_click(on_run_clicked)

# DISPLAY UI

display(title, query_input, run_button, output_area)


HTML(value='\n    <h2 style="text-align:center;">Multi-Agent Intelligent System</h2>\n    <p style="text-align…

Textarea(value='', layout=Layout(height='140px', width='100%'), placeholder='Enter your AI-related query here.…

Button(button_style='primary', description='Generate Response', icon='play', style=ButtonStyle())

Output(layout=Layout(border='1px solid black', padding='10px'))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
# convert ipynb file to HTML
!pip install nbconvert
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/is_report2.ipynb"

[NbConvertApp] Converting notebook /content/drive/MyDrive/Colab Notebooks/is_report2.ipynb to html
Traceback (most recent call last):
  File "/usr/local/bin/jupyter-nbconvert", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jupyter_core/application.py", line 284, in launch_instance
    super().launch_instance(argv=argv, **kwargs)
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 992, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 420, in start
    self.convert_notebooks()
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 597, in convert_notebooks
    self.convert_single_notebook(notebook_filename)
  File "/usr/local/lib/python3.12/dist-packages/nbconvert/nbconvertapp.py", line 563, in convert_single_notebook
    output, resources = self.export_single_notebook(
                        ^^^^^